# 🏎️ Fabric Racing Game v4 - Python Edition

**Real-time telemetry** sent automatically during gameplay!

## How to Play
1. Run Cell 1 → Configure connection
2. Run Cell 2 → Play! (use ← → arrow keys)

Telemetry is sent to Eventhouse as you race - no copy/paste needed!

In [ ]:
# ⚙️ CELL 1: CONFIGURATION
# Copy the FULL Connection String from: Eventstream → Custom Endpoint → Keys

CONNECTION_STRING = "Endpoint=sb://YOUR_NAMESPACE.servicebus.windows.net/;SharedAccessKeyName=YOUR_KEY_NAME;SharedAccessKey=YOUR_KEY;EntityPath=YOUR_EVENTHUB"
PLAYER_NAME = "Player1"

# Install SDK
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "azure-eventhub", "-q"], stdout=subprocess.DEVNULL)

# Initialize sender
from azure.eventhub import EventHubProducerClient, EventData
import json

class TelemetrySender:
    def __init__(self, conn_str):
        self.producer = None
        self.sent = 0
        self.errors = 0
        if conn_str and 'YOUR_' not in conn_str:
            try:
                self.producer = EventHubProducerClient.from_connection_string(conn_str)
                print("✅ Connected to Eventstream!")
            except Exception as e:
                print(f"⚠️ Connection failed: {e}")
        else:
            print("⚠️ Update CONNECTION_STRING above to enable telemetry.")
    
    def send(self, event):
        if self.producer:
            try:
                batch = self.producer.create_batch()
                batch.add(EventData(json.dumps(event)))
                self.producer.send_batch(batch)
                self.sent += 1
            except Exception as e:
                self.errors += 1
                if self.errors <= 3:
                    print(f"Send error: {e}")
    
    def close(self):
        if self.producer:
            self.producer.close()

sender = TelemetrySender(CONNECTION_STRING)
print(f"🏎️ Player: {PLAYER_NAME}")

In [ ]:
# 🎮 CELL 2: PLAY THE GAME!
import numpy as np
import matplotlib
matplotlib.use('module://matplotlib_inline.backend_inline')  # For Fabric notebooks
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle, Circle
from IPython.display import display, clear_output, Image
import io, time, random, uuid
from datetime import datetime, timezone

# Game constants
LEVELS = [
    {"name": "Lakehouse Lane", "target": 500, "stars": 12, "bugs": 5, "speed": 4},
    {"name": "Pipeline Pass", "target": 800, "stars": 14, "bugs": 7, "speed": 4.5},
    {"name": "Warehouse Way", "target": 1200, "stars": 16, "bugs": 9, "speed": 5},
    {"name": "Dataflow Drive", "target": 1600, "stars": 18, "bugs": 11, "speed": 5.5},
    {"name": "Notebook Narrows", "target": 2000, "stars": 20, "bugs": 14, "speed": 6},
    {"name": "Eventhouse Express", "target": 2500, "stars": 22, "bugs": 17, "speed": 6.5},
    {"name": "Shortcut Sprint", "target": 3000, "stars": 24, "bugs": 20, "speed": 7},
    {"name": "Capacity Canyon", "target": 3500, "stars": 26, "bugs": 24, "speed": 7.5},
    {"name": "OneLake Overdrive", "target": 4000, "stars": 28, "bugs": 28, "speed": 8},
    {"name": "Spark Summit", "target": 5000, "stars": 32, "bugs": 32, "speed": 9},
]

# Game state
session_id = str(uuid.uuid4())
current_level = 0
car_x = 50  # 0-100 range
score = 0
total_score = 0
lives = 3
multiplier = 1
consecutive_stars = 0
distance = 0
game_over = False
game_won = False

# Generate obstacles for a level
def generate_level(level_idx):
    level = LEVELS[level_idx]
    stars = []
    bugs = []
    for i in range(level['stars']):
        stars.append({'x': random.randint(15, 85), 'y': 200 + i * 80, 'collected': False})
    for i in range(level['bugs']):
        bugs.append({'x': random.randint(15, 85), 'y': 250 + i * 100, 'hit': False})
    return stars, bugs, level['target'], level['speed'], level['name']

stars, bugs, target_score, speed, level_name = generate_level(0)
level_length = max(s['y'] for s in stars) + 200

def send_event(event_type, extra={}):
    event = {
        'EventId': str(uuid.uuid4()),
        'Timestamp': datetime.now(timezone.utc).isoformat(),
        'GameSessionId': session_id,
        'PlayerId': PLAYER_NAME,
        'EventType': event_type,
        'Level': current_level + 1,
        'LevelName': level_name,
        'Score': score,
        'TotalScore': total_score,
        'Lives': lives,
        'Multiplier': multiplier,
        **extra
    }
    sender.send(event)

def render_frame():
    fig, (ax_game, ax_hud) = plt.subplots(1, 2, figsize=(10, 6),
        gridspec_kw={'width_ratios': [2.5, 1]})
    fig.patch.set_facecolor('#1a1a2e')
    
    # Game area
    ax_game.set_facecolor('#2d3436')
    ax_game.set_xlim(0, 100)
    ax_game.set_ylim(0, 100)
    ax_game.axis('off')
    
    # Road markings
    for y in range(0, 100, 15):
        y_pos = (y - (distance % 15)) % 100
        ax_game.plot([50, 50], [y_pos, y_pos + 8], 'w--', alpha=0.3, linewidth=2)
    
    # Road edges
    ax_game.axvline(x=10, color='#3498db', linewidth=4)
    ax_game.axvline(x=90, color='#3498db', linewidth=4)
    
    # Draw stars (visible ones)
    for star in stars:
        if not star['collected']:
            rel_y = 100 - (star['y'] - distance) / level_length * 200
            if 0 < rel_y < 100:
                ax_game.plot(star['x'], rel_y, '*', color='#ffd700', markersize=20)
    
    # Draw bugs (visible ones)
    for bug in bugs:
        if not bug['hit']:
            rel_y = 100 - (bug['y'] - distance) / level_length * 200
            if 0 < rel_y < 100:
                ax_game.plot(bug['x'], rel_y, 'o', color='#e74c3c', markersize=15)
                ax_game.text(bug['x'], rel_y, '🐛', fontsize=12, ha='center', va='center')
    
    # Draw car
    ax_game.text(car_x, 15, '🏎️', fontsize=30, ha='center', va='center')
    
    # Progress bar
    progress = min(100, distance / level_length * 100)
    ax_game.add_patch(Rectangle((92, 5), 6, 90, facecolor='#333', edgecolor='#555'))
    ax_game.add_patch(Rectangle((92, 5), 6, progress * 0.9, facecolor='#2ecc71'))
    ax_game.text(95, progress * 0.9 + 5, '🏎️', fontsize=10, ha='center')
    
    # HUD
    ax_hud.set_facecolor('#0a0a0a')
    ax_hud.axis('off')
    ax_hud.set_xlim(0, 1)
    ax_hud.set_ylim(0, 1)
    
    hud_text = (
        f'🏎️ FABRIC RACING\n'
        f'={"="*20}\n\n'
        f'Level {current_level + 1}/10\n'
        f'{level_name}\n\n'
        f'Score: {score}\n'
        f'Target: {target_score}\n'
        f'Multiplier: x{multiplier}\n\n'
        f'Lives: {"❤️" * lives}{"🖤" * (3 - lives)}\n\n'
        f'Progress: {int(progress)}%\n\n'
        f'← → to steer\n'
        f'Collect ⭐ Avoid 🐛'
    )
    ax_hud.text(0.1, 0.95, hud_text, fontfamily='monospace', fontsize=10,
                color='#00ff00', va='top', transform=ax_hud.transAxes)
    
    # Telemetry status
    ax_hud.text(0.1, 0.05, f'📡 Events: {sender.sent}', fontsize=8, color='#888',
                transform=ax_hud.transAxes)
    
    plt.tight_layout()
    
    # Save to buffer
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=80, facecolor='#1a1a2e', bbox_inches='tight')
    plt.close(fig)
    return buf.getvalue()

# Main game loop
print("🏎️ Starting Fabric Racing Game!")
print("Use keyboard: ← → to steer (click on the output first)")
print("Press 'q' to quit\n")

send_event('GameStart')

frame_count = 0
telemetry_interval = 10  # Send telemetry every N frames

# Simple auto-play simulation (since Fabric notebooks don't support real keyboard)
try:
    while not game_over and not game_won:
        # Auto-steer towards stars, away from bugs
        target_x = car_x
        for star in stars:
            if not star['collected']:
                rel_y = star['y'] - distance
                if 20 < rel_y < 150:  # Upcoming star
                    target_x = star['x']
                    break
        
        # Avoid bugs
        for bug in bugs:
            if not bug['hit']:
                rel_y = bug['y'] - distance
                if 10 < rel_y < 50 and abs(bug['x'] - car_x) < 15:
                    target_x = car_x + (15 if car_x < 50 else -15)
        
        # Move car towards target
        if car_x < target_x - 2:
            car_x = min(85, car_x + 3)
        elif car_x > target_x + 2:
            car_x = max(15, car_x - 3)
        
        # Update distance
        distance += speed
        
        # Check star collisions
        for star in stars:
            if not star['collected']:
                rel_y = star['y'] - distance
                if abs(rel_y) < 20 and abs(star['x'] - car_x) < 10:
                    star['collected'] = True
                    consecutive_stars += 1
                    if consecutive_stars % 3 == 0:
                        multiplier = min(10, multiplier + 1)
                    points = 50 * multiplier
                    score += points
                    total_score += points
                    send_event('StarCollected', {'Points': points})
        
        # Check bug collisions
        for bug in bugs:
            if not bug['hit']:
                rel_y = bug['y'] - distance
                if abs(rel_y) < 15 and abs(bug['x'] - car_x) < 8:
                    bug['hit'] = True
                    lives -= 1
                    multiplier = 1
                    consecutive_stars = 0
                    score = max(0, score - 30)
                    send_event('BugHit', {'LivesRemaining': lives})
                    if lives <= 0:
                        game_over = True
                        send_event('GameOver', {'FinalScore': total_score})
        
        # Check level complete
        if distance >= level_length:
            if score >= target_score:
                send_event('LevelComplete', {'LevelScore': score})
                current_level += 1
                if current_level >= len(LEVELS):
                    game_won = True
                    send_event('GameWon', {'FinalScore': total_score})
                else:
                    # Next level
                    score = 0
                    distance = 0
                    stars, bugs, target_score, speed, level_name = generate_level(current_level)
                    level_length = max(s['y'] for s in stars) + 200
                    send_event('LevelStart', {'TargetScore': target_score})
            else:
                lives -= 1
                if lives <= 0:
                    game_over = True
                    send_event('GameOver', {'FinalScore': total_score, 'Reason': 'TargetNotReached'})
                else:
                    # Retry level
                    score = 0
                    distance = 0
                    stars, bugs, target_score, speed, level_name = generate_level(current_level)
                    level_length = max(s['y'] for s in stars) + 200
        
        # Send periodic telemetry
        if frame_count % telemetry_interval == 0:
            send_event('Telemetry', {'CarX': car_x, 'Distance': int(distance)})
        
        # Render frame
        frame_count += 1
        if frame_count % 3 == 0:  # Render every 3rd frame for performance
            png_data = render_frame()
            clear_output(wait=True)
            display(Image(data=png_data))
        
        time.sleep(0.05)  # ~20 FPS

except KeyboardInterrupt:
    print("\n🛑 Game interrupted")

# Final summary
sender.close()
clear_output(wait=True)

if game_won:
    print("🏆 CONGRATULATIONS! YOU WON!")
    print(f"Final Score: {total_score}")
elif game_over:
    print("💀 GAME OVER")
    print(f"Final Score: {total_score}")
    print(f"Reached Level: {current_level + 1}")

print(f"\n📡 Telemetry Summary:")
print(f"   Events sent: {sender.sent}")
if sender.errors:
    print(f"   Errors: {sender.errors}")